In [ ]:
# ruff: noqa: T201, T203, ERA001, BLE001
from __future__ import annotations

import os
from pathlib import Path

if Path.cwd().name == "notebooks":
    os.chdir("..")

import logging

import pandas as pd
from dotenv import load_dotenv
from tqdm import tqdm

import src.prompts.system as system_prompts
from config.config import RUN_DIR
from config.runs_vars import SUBSET_TO_DATASET_MAP
from src.constants import PAIRS_SEPARATOR
from src.evaluate import (
    get_predictions_with_gt,
    plot_usage_histograms,
    read_run_metrics_df,
    save_analysis_results,
    store_run_metrics_df,
)
from src.formatting import (
    format_oracle_pairs_filepath,
    format_oracle_pairs_precomputed_dir,
    format_predictions_run_path,
    format_run_path,
    format_storing_pathes_from_run_path,
    format_subsets_ontologies_paths,
)
from src.LLM_servers.openai import OpenAIServer
from src.LLM_servers.qwen import QwenServer

from src.onto_access import OntologyAccess
from src.onto_object import OntologyEntryAttr
from src.processing import parallel_samples_process, save_oracle_pairs_with_prompts, try_load_precomputed_oracle_pairs
from src.prompts.prompts import (
    prompt_direct_entity,
    prompt_direct_entity_ontological,
    prompt_direct_entity_with_synonyms,
    prompt_sequential_hierarchy,
    prompt_sequential_hierarchy_ontological,
    prompt_sequential_hierarchy_with_synonyms,
)
from src.utils import read_oracle_pairs, save_run_results

pd.set_option("display.max_rows", None)
logging.getLogger().setLevel(logging.WARNING)
load_dotenv()

%load_ext autoreload
%autoreload 2

### Run experiments


In [ ]:
MAX_WORKERS = 1
MODELS = [
    # "qwen3-1.7B",
    # "qwen3-8B",
    "qwen1.5-MoE-A2.7B",
]

DATASETS_MAP = {
    # "largebio": ["fma-nci"],  # , "snomed-nci", "fma-snomed"],
    # "bioml-2024": ["omim-ordo",] #  "snomed-fma.body", "snomed-ncit.neoplas", "snomed-ncit.pharm", "ncit-doid"]
    "anatomy": ["human-mouse"],
}
EXPERIMENT_TYPE = "prompts"

from src.constants import BinaryOutputFormat, BinaryOutputFormatWithReasoning

# EXP_NAMES_MAP = {"present": BinaryOutputFormat, "absent": BinaryOutputFormatWithReasoning}
EXP_NAMES_MAP = {"present": BinaryOutputFormat}

onto_src, onto_tgt = None, None

PROMPT_FUNCTIONS_MAP = {
    prompt_function.__name__.replace("prompt_", ""): prompt_function
    for prompt_function in [
        prompt_direct_entity,
        # prompt_direct_entity_ontological,
        # prompt_direct_entity_with_synonyms,
        # prompt_sequential_hierarchy,
        # prompt_sequential_hierarchy_ontological,
        # prompt_sequential_hierarchy_with_synonyms,
    ]
}

RUNS = []

In [ ]:
llm_oracle = QwenServer(base_url="http://localhost:8000")
llm_oracle.ask_sync_question("How old are you?")

In [ ]:
for DATASET in DATASETS_MAP:
    for SUBSET in DATASETS_MAP[DATASET]:
        # Load the ontologies here, if not using precomputed prompts
        # src_onto_path, tgt_onto_path = format_subsets_ontologies_paths(DATASET, SUBSET)
        # onto_src = OntologyAccess(src_onto_path, annotate_on_init=True)
        # onto_tgt = OntologyAccess(tgt_onto_path, annotate_on_init=True)
        for exp_name in EXP_NAMES_MAP:
            if exp_name == "present" and SUBSET == "omin-ordo":
                continue
            for MODEL in MODELS:
                if MODEL.startswith("gemini"):
                    llm_oracle = (
                        OpenAIServer(
                            api_key=os.environ["GEMINI_API_KEY"],
                            base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
                            response_format=EXP_NAMES_MAP[exp_name],
                            max_tokens=1000,  # Adjust as needed
                            )
                        )
                elif MODEL.startswith("qwen"):
                    llm_oracle = QwenServer(base_url="http://localhost:8000")
                else:
                    llm_oracle = (
                        OpenAIServer(
                            api_key=os.environ["OPENAI_API_KEY"],
                            response_format=EXP_NAMES_MAP[exp_name],
                            max_tokens=1000,  # Adjust as needed
                        )
                    )

                # llm_oracle.add_system_context(system_prompts.INTUITIVE_NATURAL_LANGUAGE_JUDGEMENT_MESSAGE)

                run_path = (
                    format_predictions_run_path(DATASET, SUBSET, MODEL, EXPERIMENT_TYPE, exp_spec=exp_name)
                    if exp_name
                    else format_run_path()
                )
                print(f"{run_path=} | {DATASET=} | {SUBSET=} | {MODEL=} | {exp_name=}")

                for prompt_name, prompt_function in PROMPT_FUNCTIONS_MAP.items():
                    prediction_path, stats_path, diagram_path = format_storing_pathes_from_run_path(
                        run_path, SUBSET, MODEL, prompt_name, suffix=""
                    )
                    oracle_candidate_pairs = try_load_precomputed_oracle_pairs(
                        DATASET, SUBSET, prompt_function, suffix=""
                    )
                    results, tokens_usage, confidences = parallel_samples_process(
                        oracle_candidate_pairs, llm_oracle, onto_src, onto_tgt, MODEL, MAX_WORKERS, prompt_function
                    )

                    save_run_results(results, prediction_path, columns=["Source", "Target", "Prediction", "Confidence"])
                    plot_usage_histograms(
                        tokens_usage, confidences, do_plot=False, do_print=False, suptitle=prompt_name
                    )
                    try:
                        predictions = get_predictions_with_gt(run_path, DATASET, SUBSET, MODEL, prompt_name, suffix="")
                        save_analysis_results(
                            predictions,
                            print_results=False,
                            plot_confusion_matrix=False,
                            subtitle=f"{SUBSET}: {MODEL} {prompt_name} | ",
                            cm_save_path=diagram_path,
                            stats_path=stats_path,
                        )
                    except Exception as e:
                        print(f"Error: {e}")

                store_run_metrics_df(PROMPT_FUNCTIONS_MAP, run_path, DATASET, SUBSET, MODEL)

